In [13]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    # 5% szans na oszusta
    if random.random() < 0.05:
        amount = round(random.uniform(3001.0, 5000.0), 2)
        category = 'elektronika'
        hour = random.randint(0, 5)
    else:
        # 95% przypadków normalny klient
        amount = round(random.uniform(5.0, 5000.0), 2)
        category = random.choice(kategorie)
        hour = random.randint(6, 23)
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': amount,
        'store': random.choice(sklepy),
        'category': category,
        'hour': hour,
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Overwriting producer.py


In [3]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Konsument uruchomiony... Czekam na duże transakcje (> 1000 PLN)")

for message in consumer:
    tx = message.value
    
    if tx['amount'] > 1000:
        print(f"ALERT: Duża transakcja! ID: {tx['tx_id']} | Kwota: {tx['amount']:.2f} PLN | Sklep: {tx['store']}")
    else:
        pass

Writing consumer_filter.py


In [4]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Konsument - analiza ryzyka")

for message in consumer:
    tx = message.value

    if tx['amount'] > 3000:
        risk_level = "HIGH"
    elif tx['amount'] > 1000:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"
    
    # nowe pole do słownika
    tx['risk_level'] = risk_level
    
    # Wyświetlamy efekt
    color = "🔴" if risk_level == "HIGH" else ("🟡" if risk_level == "MEDIUM" else "🟢")
    print(f"{color} [{risk_level}] ID: {tx['tx_id']} | Kwota: {tx['amount']:.2f} PLN | Sklep: {tx['store']}")

Writing consumer_enrich.py


In [5]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter, defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()          # ilość transakcji
total_amount = defaultdict(float) # wartość transakcji
msg_count = 0                     # ilość przetworzonych wiadomości

print("Konsument agregujący")

for message in consumer:
    tx = message.value
    store = tx['store']
    amount = tx['amount']

    store_counts[store] += 1
    total_amount[store] += amount
    msg_count += 1
    
    # Raportowanie co 10 wiadomości
    if msg_count % 10 == 0:
        print(f"\n Stan po {msg_count} transakcjach")
        print(f"{'Sklep':<12} | {'Ilość':<6} | {'Suma (PLN)':<12}")
        print("-" * 30)
        
        # Pętla po wszystkich sklepach
        for s in store_counts:
            print(f"{s:<12} | {store_counts[s]:<6} | {total_amount[s]:.2f}")

Writing consumer_count.py


In [8]:
# funkcja

In [7]:
from datetime import datetime

def score_transaction(tx):
    score = 0
    rules = []
    
    # R1: Kwota > 3000 (+3 punkty)
    if tx['amount'] > 3000:
        score += 3
        rules.append('R1')
        
    # R2: Kategoria 'elektronika' i kwota > 1500 (+2 punkty)
    if tx['category'] == 'elektronika' and tx['amount'] > 1500:
        score += 2
        rules.append('R2')
        
    # R3: Godzina nocna (< 6) (+2 punkty)
    tx_time = datetime.fromisoformat(tx['timestamp'])
    if tx_time.hour < 6:
        score += 2
        rules.append('R3')
        
    return score, rules

# Test
test_tx = {
    'tx_id': 'TX999', 
    'amount': 4500.0, 
    'category': 'elektronika', 
    'timestamp': '2026-04-01T03:15:00'
}

total_score, broken_rules = score_transaction(test_tx)

print(f"Wynik punktowy: {total_score}")
print(f"Naruszone reguły: {broken_rules}")
if total_score >= 3:
    print("STATUS: TRANSAKCJA PODEJRZANA!")

Wynik punktowy: 7
Naruszone reguły: ['R1', 'R2', 'R3']
STATUS: TRANSAKCJA PODEJRZANA!


In [12]:
%%file scoring_consumer.py
from kafka import KafkaConsumer, KafkaProducer
import json
from datetime import datetime

# Konfiguracja Konsumenta (czyta z 'transactions')
consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='scoring-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

# Konfiguracja Producenta (wysyła do 'alerts')
alert_producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

def score_transaction(tx):
    score = 0
    rules = []
    
    # R1: kwota > 3000 (+3)
    if tx['amount'] > 3000:
        score += 3
        rules.append('R1')
        
    # R2: elektronika i kwota > 1500 (+2)
    if tx['category'] == 'elektronika' and tx['amount'] > 1500:
        score += 2
        rules.append('R2')
        
    # R3: godzina < 6 (noc) (+2)
    tx_time = datetime.fromisoformat(tx['timestamp'])
    if tx_time.hour < 6:
        score += 2
        rules.append('R3')
        
    return score, rules

print("System monitoringu uruchomiony...")

for message in consumer:
    tx = message.value
    
    # Scoring
    total_score, broken_rules = score_transaction(tx)
    
    # Jeśli transakcja jest podejrzana (score >= 3)
    if total_score >= 3:
        # Wzbogacamy dane o wyniki analizy
        tx['total_score'] = total_score
        tx['broken_rules'] = broken_rules
        
        # Producent 'alert_producer' wysyła informację do nowego tematu Kafki
        alert_producer.send('alerts', value=tx)
        
        print(f"ALERT! Podejrzana transakcja {tx['tx_id']}: {total_score} pkt. Reguły: {broken_rules}")

Overwriting scoring_consumer.py
